# RipeSense banana model: train our own ripeness classifier (Colab)

Trains **one model** for the AiNex fruit-sorting robot:

| Class | Robot action |
|---|---|
| `ripe` | pick → **basket** |
| `rotten` | pick → **bin** |
| `overripe` | pick → bin (configurable) |
| `unripe` | leave it |
| `not_banana` | ignore (people, phones, boxes, other fruit…) |

**Data**, downloaded here from the original public sources (CC BY 4.0):
* *Annotated Banana Ripeness Stages*, Mendeley [56c75hvm7r v2](https://data.mendeley.com/datasets/56c75hvm7r/2): bounding boxes, unripe/ripe/overripe; we crop each banana.
* *Fresh and Rotten Fruits*, Mendeley [bdd69gyhv8 v1](https://data.mendeley.com/datasets/bdd69gyhv8/1): `rotten` bananas, and other fruit as `not_banana`.
* **Our own photos through the robot's camera** (`own_data.zip`, optional but strongly recommended): collected with `run.sh collect` on the robot. These are reported as a separate test score.

Model: MobileNetV3-Small (ImageNet init), fine-tuned. Exported to ONNX and checked against OpenCV DNN, which is what runs on the robot's Pi 4.

**Runtime → Change runtime type → T4 GPU**, then Runtime → Run all. About 15–25 min total (downloads ≈ 5 GB).

In [ ]:
# ---------------- settings ----------------
import os, sys, random, numpy as np, torch
EPOCHS          = 15        # 15 is plenty; val score picks the best epoch
BATCH           = 64
LR              = 1e-3
IMG_SIZE        = 320       # crops are stored at this max side
MAX_ROTTEN      = 700       # cap rotten bananas (Fresh&Rotten has ~1.3k)
MAX_NOT_BANANA  = 1600      # cap other-fruit "not_banana" images
OWN_DATA_ZIP    = "/content/own_data.zip" if os.path.isdir("/content") else "own_data.zip"   # upload here (Files panel) or put it in Drive and set the path
USE_DRIVE       = True      # cache the 5 GB of downloads + save results in Google Drive
SEED            = 0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to GPU")
IN_COLAB = "google.colab" in sys.modules
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ripesense_train"
else:
    WORK = "/content/ripesense_train" if IN_COLAB else os.path.abspath("ripesense_train")
RAW, DATA, OUT = f"{WORK}/raw", ("/content/data" if IN_COLAB else f"{WORK}/data"), f"{WORK}/output"
for d in (RAW, OUT): os.makedirs(d, exist_ok=True)
print("work dir:", WORK)

In [ ]:
# ---------------- download (skipped if already cached) ----------------
import subprocess
SOURCES = {
    "BANANA_DATASET75.zip": "https://data.mendeley.com/public-files/datasets/56c75hvm7r/files/1654da99-8c51-49da-b674-a272a8afcccb/file_downloaded",
    "FreshRotten_Original.zip": "https://data.mendeley.com/public-files/datasets/bdd69gyhv8/files/de93ba06-6a58-45e3-913d-837b2ae52acb/file_downloaded",
}
EXPECTED = {"BANANA_DATASET75.zip": 2322506557, "FreshRotten_Original.zip": 2794170228}
for name, url in SOURCES.items():
    path = f"{RAW}/{name}"
    if os.path.exists(path) and os.path.getsize(path) == EXPECTED[name]:
        print("cached:", name); continue
    print("downloading", name, f"({EXPECTED[name]/1e9:.1f} GB)…")
    subprocess.run(["wget", "-q", "-c", "-O", path, url], check=True)
    assert os.path.getsize(path) == EXPECTED[name], f"{name}: size mismatch, delete it and re-run"
print("downloads OK")

In [ ]:
# ---------------- build the 5-class dataset ----------------
# Same cropping/splitting rules as ripesense/models/prep_datasets.py.
import hashlib, re, shutil, zipfile, yaml, cv2
from collections import Counter
from pathlib import Path, PurePosixPath

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLITS = {"train": "train", "valid": "val", "val": "val", "validation": "val", "test": "test"}
CLASSES = ["not_banana", "overripe", "ripe", "rotten", "unripe"]   # alphabetical = ImageFolder order
FRUITS = ["apple", "banana", "orange", "grape", "guava", "jujube", "pomegranate", "strawberry", "lime",
          "mango", "tomato", "avocado", "kiwi", "lemon", "peach", "pear", "papaya", "potato", "cucumber"]
shutil.rmtree(DATA, ignore_errors=True)
counts = Counter()

def decode(z, n): return cv2.imdecode(np.frombuffer(z.read(n), np.uint8), cv2.IMREAD_COLOR)
def shrink(img, size=IMG_SIZE):
    s = size / max(img.shape[:2])
    return cv2.resize(img, (int(img.shape[1]*s), int(img.shape[0]*s)), interpolation=cv2.INTER_AREA) if s < 1 else img
def hash_split(key, val=0.1, test=0.1):
    h = int(hashlib.md5(key.encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    return "test" if h < test else "val" if h < test + val else "train"
def save(img, split, cls, name, source):
    d = Path(DATA) / split / cls; d.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(d / f"{source}__{name}.jpg"), shrink(img), [cv2.IMWRITE_JPEG_QUALITY, 92])
    counts[(source, split, cls)] += 1
def stage_name(name):
    s = re.sub(r"[^a-z]", "", name.lower())
    if "over" in s: return "overripe"
    if "unripe" in s or "green" in s or "raw" in s: return "unripe"
    if "rot" in s or "spoil" in s: return "rotten"
    if "ripe" in s: return "ripe"
    return None

# 1) banana stages: crop every labelled banana, keep the dataset's own train/val/test split
z = zipfile.ZipFile(f"{RAW}/BANANA_DATASET75.zip"); names = z.namelist()
classes = None
for y in [n for n in names if n.lower().endswith(".yaml")]:
    d = yaml.safe_load(z.read(y))
    if isinstance(d, dict) and "names" in d:
        n = d["names"]; classes = [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n); break
classes = classes or ["unripe", "ripe", "overripe"]       # checked against the file names below
print("banana dataset classes:", classes)
imgs = {PurePosixPath(n).stem: n for n in names if PurePosixPath(n).suffix.lower() in IMG_EXT}
labels = [n for n in names if n.endswith(".txt") and "labels" in PurePosixPath(n).parts]
mismatch = 0
for ln in labels:
    stem = PurePosixPath(ln).stem
    if stem not in imgs: continue
    split = next((SPLITS[p.lower()] for p in reversed(PurePosixPath(ln).parts) if p.lower() in SPLITS), "train")
    rows = [r.split() for r in z.read(ln).decode().splitlines() if len(r.split()) >= 5]
    if not rows: continue
    img = decode(z, imgs[stem])
    if img is None: continue
    H, W = img.shape[:2]
    for k, p in enumerate(rows):
        c = int(float(p[0]))
        if len(p) > 5:
            xs, ys = [float(v) for v in p[1::2]], [float(v) for v in p[2::2]]
            cx, cy, bw, bh = (min(xs)+max(xs))/2, (min(ys)+max(ys))/2, max(xs)-min(xs), max(ys)-min(ys)
        else:
            cx, cy, bw, bh = map(float, p[1:5])
        bw, bh = bw * 1.24, bh * 1.24                       # 12 % padding each side
        x0, x1 = int(max(0, (cx-bw/2)*W)), int(min(W, (cx+bw/2)*W))
        y0, y1 = int(max(0, (cy-bh/2)*H)), int(min(H, (cy+bh/2)*H))
        if min(x1-x0, y1-y0) < 48: continue
        cls = stage_name(classes[c])
        fn = stage_name(stem)
        if fn and fn != cls: mismatch += 1
        if cls: save(img[y0:y1, x0:x1], split, cls, f"{stem}_{k}", "stages")
print("label/filename disagreements:", mismatch)

# 2) Fresh & Rotten: rotten bananas -> rotten ; other fruit (fresh+rotten) -> not_banana ; fresh bananas unused (no stage label)
z = zipfile.ZipFile(f"{RAW}/FreshRotten_Original.zip")
members = sorted([n for n in z.namelist() if PurePosixPath(n).suffix.lower() in IMG_EXT],
                 key=lambda n: hashlib.md5(n.encode()).hexdigest())
n_rot = n_nb = 0
per_fruit = Counter()
for n in members:
    pl = n.lower()
    if "mixed" in pl or "augment" in pl: continue
    rot, fresh = any(w in pl for w in ("rotten", "bad", "spoil", "stale")), any(w in pl for w in ("fresh", "good"))
    if rot == fresh: continue
    fruit = next((f for f in FRUITS if f in pl.replace("pineapple", "")), None)
    if fruit is None: continue
    if fruit == "banana":
        if not rot or n_rot >= MAX_ROTTEN: continue
        cls = "rotten"; n_rot += 1
    else:
        if n_nb >= MAX_NOT_BANANA or per_fruit[fruit] >= MAX_NOT_BANANA // 6: continue
        cls = "not_banana"; n_nb += 1; per_fruit[fruit] += 1
    img = decode(z, n)
    if img is None: continue
    save(img, hash_split("fr/" + n), cls, hashlib.md5(n.encode()).hexdigest()[:12], "freshrotten")

# 3) our own robot-camera photos: own_data.zip with folders <class>/*.jpg (any subset of the 5 classes)
if os.path.exists(OWN_DATA_ZIP):
    z = zipfile.ZipFile(OWN_DATA_ZIP)
    for n in z.namelist():
        parts = PurePosixPath(n).parts
        if PurePosixPath(n).suffix.lower() not in IMG_EXT or len(parts) < 2: continue
        cls = parts[-2].lower()
        if cls not in CLASSES: continue
        img = decode(z, n)
        if img is None: continue
        # split by capture session (file prefix) when present so near-identical frames don't leak across splits
        key = PurePosixPath(n).stem.split("__")[0]
        save(img, hash_split("own/" + key, 0.15, 0.2), cls, PurePosixPath(n).stem, "own")
else:
    print("no own_data.zip found: training on public data only (robot-camera score will be missing)")

import pandas as pd
df = pd.Series(counts).rename_axis(["source", "split", "class"]).unstack("class", fill_value=0)
print(df.reindex(columns=CLASSES, fill_value=0))
for s in ("train", "val", "test"):
    missing = [c for c in CLASSES if not (Path(DATA)/s/c).exists()]
    assert not missing, f"{s} split is missing classes {missing}"

In [ ]:
# ---------------- look at a few training crops ----------------
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(CLASSES), 8, figsize=(16, 2.2*len(CLASSES)))
for r, c in enumerate(CLASSES):
    fs = sorted((Path(DATA)/"train"/c).glob("*.jpg")); random.shuffle(fs)
    for k, ax in enumerate(axes[r]):
        ax.axis("off")
        if k < len(fs): ax.imshow(cv2.cvtColor(cv2.imread(str(fs[k])), cv2.COLOR_BGR2RGB))
    axes[r][0].set_title(c, loc="left")
plt.tight_layout(); plt.show()

In [ ]:
# ---------------- train ----------------
import time, json, torch.nn as nn
from PIL import Image
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
dev = "cuda" if torch.cuda.is_available() else "cpu"

class BackgroundSwap:
    """Studio photos have white backgrounds; the robot sees floors. Replace light,
    unsaturated background with random grey/dark so the model ignores it."""
    def __init__(self, p=0.5): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        a = np.asarray(img.convert("RGB")).astype(np.int16)
        mx, mn = a.max(2), a.min(2)
        bg = ((mx - mn) / np.maximum(mx, 1) < 0.18) & (mx > 140)
        if bg.mean() < 0.08: return img
        v, t = random.randint(8, 200), random.randint(-6, 6)
        col = np.array((v + t, v, v - t), np.int16)
        a = np.where(bg[..., None], np.clip(col + np.random.randint(-12, 13, a.shape, dtype=np.int16), 0, 255), a)
        return Image.fromarray(a.astype(np.uint8))

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_train = transforms.Compose([
    BackgroundSwap(0.5),
    transforms.RandomResizedCrop(224, scale=(0.55, 1.0), ratio=(0.6, 1.6)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(25),
    # robot camera: dimmer, blurrier, lower contrast than phone photos
    transforms.ColorJitter(brightness=0.4, contrast=0.35, saturation=0.25, hue=0.02),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 1.6))], p=0.3),
    transforms.ToTensor(), norm, transforms.RandomErasing(p=0.2, scale=(0.02, 0.12))])
tf_eval = transforms.Compose([transforms.Resize(240), transforms.CenterCrop(224), transforms.ToTensor(), norm])

tr = datasets.ImageFolder(f"{DATA}/train", tf_train)
va = datasets.ImageFolder(f"{DATA}/val", tf_eval)
te = datasets.ImageFolder(f"{DATA}/test", tf_eval)
assert tr.classes == va.classes == te.classes == CLASSES, tr.classes
cnt = np.bincount(tr.targets, minlength=len(CLASSES))
print("train counts", dict(zip(CLASSES, cnt.tolist())))
sampler = WeightedRandomSampler(torch.as_tensor((1.0 / cnt)[tr.targets], dtype=torch.double), len(tr), replacement=True)
tl = DataLoader(tr, batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)

net = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
net.classifier[3] = nn.Linear(net.classifier[3].in_features, len(CLASSES))
net.to(dev)
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(tl))
lossf = nn.CrossEntropyLoss(label_smoothing=0.05)

def predict(ds):
    net.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in DataLoader(ds, batch_size=128, num_workers=2):
            ps.append(net(x.to(dev)).softmax(1).cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)
def scores(y, p):
    cm = np.zeros((len(CLASSES),) * 2, int)
    for t, q in zip(y, p.argmax(1)): cm[t, q] += 1
    rec, prec = cm.diagonal() / np.maximum(1, cm.sum(1)), cm.diagonal() / np.maximum(1, cm.sum(0))
    f1 = np.where(rec + prec > 0, 2 * rec * prec / np.maximum(1e-9, rec + prec), 0)
    present = cm.sum(1) > 0
    return {"n": int(cm.sum()), "accuracy": float(np.trace(cm) / max(1, cm.sum())),
            "macro_f1": float(f1[present].mean()) if present.any() else 0.0, "confusion": cm.tolist()}

best, best_state, hist = -1, None, []
for ep in range(EPOCHS):
    net.train(); t0 = time.time(); tot = n = 0
    for x, y in tl:
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        opt.zero_grad(); loss = lossf(net(x), y); loss.backward(); opt.step(); sched.step()
        tot += loss.item() * len(y); n += len(y)
    m = scores(*predict(va)); hist.append((tot / n, m["accuracy"], m["macro_f1"]))
    print(f"epoch {ep+1:2d}/{EPOCHS}  loss {tot/n:.3f}  val acc {m['accuracy']:.3f}  val macro-F1 {m['macro_f1']:.3f}  ({time.time()-t0:.0f}s)")
    if m["macro_f1"] >= best:
        best, best_state = m["macro_f1"], {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
net.load_state_dict(best_state); net.to(dev)
print("best val macro-F1", round(best, 4))

In [ ]:
# ---------------- test: overall, public data, and OUR robot photos separately ----------------
y, p = predict(te)
src = np.array([Path(s).name.split("__")[0] for s, _ in te.samples])
metrics = {"classes": CLASSES, "epochs": EPOCHS, "train_counts": dict(zip(CLASSES, cnt.tolist())),
           "val": scores(*predict(va)), "test": scores(y, p),
           "test_by_source": {s: scores(y[src == s], p[src == s]) for s in sorted(set(src))},
           "sources": {"stages": "Mendeley 56c75hvm7r v2 (CC BY 4.0)", "freshrotten": "Mendeley bdd69gyhv8 v1 (CC BY 4.0)",
                       "own": "our photos through the AiNex robot camera"}}
def show(title, m):
    print(f"\n{title}: n={m['n']}  accuracy {m['accuracy']:.3f}  macro-F1 {m['macro_f1']:.3f}")
    print("   rows = true, cols = predicted:", CLASSES)
    for c, row in zip(CLASSES, m["confusion"]):
        if sum(row): print(f"   {c:11s} {row}")
show("TEST (all)", metrics["test"])
for s, m in metrics["test_by_source"].items(): show(f"TEST [{s}]", m)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot([h[0] for h in hist], label="train loss"); ax[0].plot([h[2] for h in hist], label="val macro-F1"); ax[0].legend(); ax[0].set_xlabel("epoch")
cm = np.array(metrics["test"]["confusion"]); ax[1].imshow(cm, cmap="Greens")
ax[1].set_xticks(range(len(CLASSES)), CLASSES, rotation=30); ax[1].set_yticks(range(len(CLASSES)), CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)): ax[1].text(j, i, cm[i, j], ha="center", va="center")
ax[1].set_title(f"test confusion (acc {metrics['test']['accuracy']:.3f})"); ax[1].set_xlabel("predicted"); ax[1].set_ylabel("true")
plt.tight_layout(); plt.savefig(f"{OUT}/training_report.png", dpi=120); plt.show()

# hardest mistakes, to see what the model gets wrong
wrong = np.where(p.argmax(1) != y)[0]; wrong = wrong[np.argsort(-p[wrong].max(1))][:12]
if len(wrong):
    fig, axes = plt.subplots(1, len(wrong), figsize=(2 * len(wrong), 2.6))
    for ax, i in zip(np.atleast_1d(axes), wrong):
        ax.imshow(Image.open(te.samples[i][0])); ax.axis("off")
        ax.set_title(f"true {CLASSES[y[i]]}\npred {CLASSES[p[i].argmax()]}", fontsize=8)
    plt.show()

In [ ]:
# ---------------- export ONNX + check it matches what the robot will run (OpenCV DNN) ----------------
net.cpu().eval()
onnx_path = f"{OUT}/banana_model.onnx"
torch.onnx.export(net, torch.randn(1, 3, 224, 224), onnx_path, input_names=["input"], output_names=["logits"],
                  opset_version=17, dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, dynamo=False)
json.dump(CLASSES, open(f"{OUT}/banana_model_classes.json", "w"))
json.dump(metrics, open(f"{OUT}/banana_model_metrics.json", "w"), indent=1)

# robot preprocessing (fruit_detector_node.py Classifier.prepare): short side 240, centre-crop 224, ImageNet norm
MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)
def robot_prepare(bgr):
    h, w = bgr.shape[:2]; s = 240.0 / min(h, w)
    img = cv2.resize(bgr, (max(224, round(w * s)), max(224, round(h * s))), interpolation=cv2.INTER_AREA)
    h, w = img.shape[:2]; y0, x0 = (h - 224) // 2, (w - 224) // 2
    x = cv2.cvtColor(img[y0:y0+224, x0:x0+224], cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return ((x - MEAN) / STD).transpose(2, 0, 1)[None].astype(np.float32)
cvnet = cv2.dnn.readNetFromONNX(onnx_path)
idx = random.sample(range(len(te)), min(300, len(te))); agree = 0; maxdiff = 0
for i in idx:
    x = robot_prepare(cv2.imread(te.samples[i][0]))
    cvnet.setInput(x); a = cvnet.forward()[0]
    with torch.no_grad(): b = net(torch.from_numpy(x))[0].numpy()
    agree += a.argmax() == b.argmax(); maxdiff = max(maxdiff, float(np.abs(a - b).max()))
print(f"OpenCV-DNN vs PyTorch on {len(idx)} test crops: same class {agree}/{len(idx)}, max logit diff {maxdiff:.4f}")
assert agree >= 0.99 * len(idx), "ONNX export does not match. Do not deploy"

zbase = "/content/banana_model" if os.path.isdir("/content") else f"{WORK}/banana_model"
shutil.make_archive(zbase, "zip", OUT)
print("saved to", OUT, "and", zbase + ".zip")
try:
    from google.colab import files; files.download(zbase + ".zip")
except Exception as e:
    print("download it from the Files panel:", e)

## Put it on the robot (from the laptop)

```bash
unzip banana_model.zip -d banana_model
scp banana_model/banana_model.onnx banana_model/banana_model_classes.json banana_model/banana_model_metrics.json \
    ubuntu@10.42.0.151:ripesense_models/
ssh ubuntu@10.42.0.151 '~/ros_ws/src/ainex_fruit_pick/run.sh detect'
```
The detector uses `banana_model.onnx` automatically when it is present. Watch it on **http://localhost:5001/robot**.

## For the pitch
Quote `banana_model_metrics.json`: `test` (all held-out images), and especially `test_by_source.own`
(our robot-camera photos, never seen in training). Public-data accuracy is on studio photos; the robot
score is the honest real-world number.